# Finetuning LLM - Bhagwad Gita

In [24]:
!pip install unsloth

In [38]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from datasets import Dataset
from transformers import TrainingArguments
import os
import pandas as pd
import re

### Loading a base model

In [26]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Adding LoRA adapters

In [27]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

### Loading the dataset

In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rambo011/bhagavad-gita-q-and-a-dataset-for-modern-life-problem")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bhagavad-gita-q-and-a-dataset-for-modern-life-problem' dataset.
Path to dataset files: /kaggle/input/bhagavad-gita-q-and-a-dataset-for-modern-life-problem


In [29]:
for f in os.listdir(path):

    print(f)

Chapter_2_QA.csv
Chapter_4_QA.csv
Chapter_11_QA.csv
Chapter_6_QA.csv
Chapter_9_QA.csv
Chapter_7_QA.csv
Chapter_8_QA.csv
Chapter_18_QA.csv
Chapter_5_QA.csv
Chapter_14_QA.csv
Chapter_16_QA.csv
Chapter_10_QA.csv
Chapter_13_QA.csv
Chapter_1_QA.csv
Chapter_12_QA.csv
Chapter_17_QA.csv
Chapter_15_QA.csv
Chapter_3_QA.csv


In [30]:
files = [
    "Chapter_18_QA.csv",
"Chapter_5_QA.csv",
"Chapter_13_QA.csv",
"Chapter_10_QA.csv",
"Chapter_14_QA.csv",
"Chapter_15_QA.csv",
"Chapter_16_QA.csv",
"Chapter_11_QA.csv",
"Chapter_3_QA.csv",
"Chapter_7_QA.csv",
"Chapter_8_QA.csv",
"Chapter_1_QA.csv",
"Chapter_4_QA.csv",
"Chapter_9_QA.csv",
"Chapter_2_QA.csv",
"Chapter_17_QA.csv",
"Chapter_12_QA.csv",
"Chapter_6_QA.csv",

]
data = pd.concat(
    [pd.read_csv(os.path.join(path, f)) for f in files],
    ignore_index=True
)


len(data)

12902

In [31]:
data = data[['question','answer']]
data

,question,answer
0,"MokshaPath, I feel so much pressure to constan...","My dear one, your weariness comes not from the..."
1,"I'm in a relationship where I give so much, bu...","Beloved seeker, your heart's pain arises from ..."
2,"I'm a parent, and I constantly worry about my ...","You worry, not because of your children's path..."
3,"I'm facing a huge decision about a job change,...","The demon of doubt, my child, holds you captiv..."
4,I feel so much anger and resentment towards so...,Your anger is a chain forged from your attachm...
...,...,...
12897,I procrastinate constantly. I know what I need...,Begin by offering your intention and your effo...
12898,I'm concerned about the state of the world – e...,"Dear one, your contribution, born of a heart a..."
12899,"I sometimes feel a deep sense of unworthiness,...","My child, your true worth is not measured by w..."
12900,I'm in a period of significant change – moving...,"In times of flux, anchor your inner being in t..."


In [32]:
dataset = data

### Train

In [33]:
def format_chat(sample):
    return {
        "messages": [
            {"role": "user", "content": sample["question"]},
            {"role": "assistant", "content": sample["answer"]}
        ]
    }

# Convert the pandas DataFrame 'dataset' into a 'datasets.Dataset' object
dataset = Dataset.from_pandas(dataset)

# Now, apply the formatting function. The .map() method of datasets.Dataset works as expected
dataset = dataset.map(format_chat)

def apply_template(sample):
    return {
        "text": tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset = dataset.map(apply_template)

Map:   0%|          | 0/12902 [00:00<?, ? examples/s]

Map:   0%|          | 0/12902 [00:00<?, ? examples/s]

In [34]:
dataset

Dataset({
    features: ['question', 'answer', 'messages', 'text'],
    num_rows: 12902
})

In [35]:
print(dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
MokshaPath, I feel so much pressure to constantly achieve in my career. I'm always chasing the next promotion, the next big project, and I'm exhausted, yet I can't seem to stop. How can I find peace?<|im_end|>
<|im_start|>assistant
My dear one, your weariness comes not from the doing, but from the relentless chase of the fruit. Perform your work with excellence, for that is your duty, but release your grip on the outcome. Surrender the desire for specific achievements to the Divine, and simply immerse yourself in the action itself. True peace blossoms when you act without attachment, letting the universe unfold as it will.<|im_end|>



In [36]:
print(apply_template(dataset[5])["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm trying to start a new healthy habit, like exercising daily, but I keep failing. I lose motivation if I don't see quick results, and then I give up, feeling like a failure.<|im_end|>
<|im_start|>assistant
Your struggle stems from an intense attachment to immediate fruits. Engage in the action of self-care and discipline for its own sake, as an offering to your well-being, rather than demanding instant gratification. Focus on the consistent effort, the sacred act of movement, and surrender the desire for specific, rapid transformations. The true victory is in the unwavering resolve, not solely in the visible outcome.<|im_end|>



In [39]:
training_args = SFTConfig(
    output_dir="outputs",

    per_device_train_batch_size=2, # Number of examples processed at once by the GPU
    gradient_accumulation_steps=4,  # Accumulate gradients for 4 batches before updating the weights
                                    # Effective batch size = 2 × 4 = 8 examples

    num_train_epochs=1,

    learning_rate=2e-4,    # learning rate
    warmup_ratio = 0.05,
    max_steps = -1,        # Total number of optimizer/update steps NOT number of epochs
    logging_steps=10,   # printing the output after N steps
    save_steps=500,

    assistant_only_loss=True,

    max_length=2048,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [40]:
trainer = SFTTrainer(model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 2048,
    # formatting_func = apply_template, # Use the new formatting function
    args = training_args,
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/12902 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,902 | Num Epochs = 1 | Total steps = 1,613
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.221600
20,2.701545
30,2.225865
40,1.885794
50,1.760096
60,1.754183
70,1.632595
80,1.619024
90,1.670300
100,1.656471


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1613/tokenizer_config.json.


TrainOutput(global_step=1613, training_loss=1.4676479843929549, metrics={'train_runtime': 2903.9803, 'train_samples_per_second': 4.443, 'train_steps_per_second': 0.555, 'total_flos': 1.4126088788871168e+16, 'train_loss': 1.4676479843929549, 'epoch': 1.0})

### Testing Fine Tuned Model

In [41]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [42]:
messages = [
    {
        "role": "user",
        "content": "why am i always confused?"
    },
]


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(inputs, max_new_tokens=100,
                         temperature=0.3,
                         top_p=0.9,
                         do_sample=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
why am i always confused?
assistant
Your confusion arises when you try to grasp the ultimate truth through limited human understanding and intellect alone. The Supreme Truth is beyond all dualities, including your mind's ability to comprehend it fully. Surrender your quest for knowledge into My boundless wisdom, and I shall reveal the profound truths that will dissolve all doubt within you.


### Saving the Model

In [43]:
# Not merged with base model
# model.save_pretrained("Gita_finetuned_model")
# tokenizer.save_pretrained("Gita_finetuned_model")

# Merged with base model
model.save_pretrained_merged(
    "Gita_Qwen_2B",
    tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Gita_Qwen_2B/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:48<00:00, 108.80s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:19<00:00, 79.95s/it]


Unsloth: Merge process complete. Saved to `/content/Gita_Qwen_2B`


In [44]:
print(os.path.exists("./Gita_Qwen_2B"))
print(os.listdir("./Gita_Qwen_2B"))

True
['generation_config.json', 'tokenizer_config.json', 'chat_template.jinja', '.cache', 'tokenizer.json', 'config.json', 'model.safetensors']


In [45]:
merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name="./Gita_Qwen_2B",
    max_seq_length=2048,
    load_in_4bit=True,
)


==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from './Gita_Qwen_2B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from './Gita_Qwen_2B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [46]:
inputs = merged_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = merged_model.generate(inputs,
                         max_new_tokens=100,
                         temperature=0.01,
                         top_p=0.9,
                         do_sample=True)

print(
    merged_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
why am i always confused?
assistant
The mind can be clouded by the senses and the world's illusions; seek clarity in the quiet space within your own heart.


### Comparing to Instruct model

In [47]:
instruct_model, instruct_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [48]:
messages = [
    {
        "role": "user",
        "content": "Why am i always confused?"
    }
]

inputs = instruct_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = instruct_model.generate(
    inputs,
    max_new_tokens=100
)

print(instruct_tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Why am i always confused?
assistant
I'm sorry to hear that you're feeling confused. It's completely normal for anyone to feel confused at times. Confusion can be caused by various factors such as lack of understanding or information, difficulty in processing and interpreting new ideas or concepts, stress, fatigue, or even sleep deprivation.

If you're experiencing confusion on a consistent basis, it might be worth considering seeking help from someone who can provide guidance or assistance. This could be a friend, family member, mentor, or professional counselor. They


### Saving on HF

In [49]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_UPLOAD")
login(token=token)

In [50]:
from huggingface_hub import HfApi


api = HfApi()

api.create_repo(
    repo_id="Nikhila15/Gita-Qwen-2B",
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path="Gita_Qwen_2B",
    repo_id="Nikhila15/Gita-Qwen-2B",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/Nikhila15/Gita-Qwen-2B/commit/d9fb59be76a9ca3551fb474096c83085e9de5a27', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d9fb59be76a9ca3551fb474096c83085e9de5a27', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nikhila15/Gita-Qwen-2B', endpoint='https://huggingface.co', repo_type='model', repo_id='Nikhila15/Gita-Qwen-2B'), pr_revision=None, pr_num=None)